In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA

from data_loaders.synthetic import TaskTrainedRNNDatasetLoader

In [ ]:
# # BlockBandit
# df_fp = '/root/capsule/code/TaskTrainedRNNData-logs_BlockBandit2ArmCoupledEasy-v1-noisy_rl2_10__16:07_13:48:09.pkl'
# RandomWalkDF
df_fp = '/root/capsule/code/TaskTrainedRNNData-logs_RandomWalkDF-v0-noisy_rl2_10__28:09_01:45:33.pkl'


df = pd.read_pickle(df_fp)
print(len(df))
df.head()

In [ ]:
# # rename columns if needed
# df_renamed = df.rename(columns={'action': 'animal_response', 'reward': 'rewarded'})
# df_renamed.to_pickle(df_fp)

In [ ]:
# # generate combined dataset: DONE on 260116

# # BlockBandit
# df_fp_block = '/root/capsule/code/TaskTrainedRNNData-logs_BlockBandit2ArmCoupledEasy-v1-noisy_rl2_10__16:07_13:48:09.pkl'
# # RandomWalkDF
# df_fp_rw = '/root/capsule/code/TaskTrainedRNNData-logs_RandomWalkDF-v0-noisy_rl2_10__28:09_01:45:33.pkl'

# df_block = pd.read_pickle(df_fp_block)
# df_rw = pd.read_pickle(df_fp_rw)

# # check unique session index
# df_rw.ses_idx.unique()

# # shift session index
# df_rw['ses_idx'] = df_rw['ses_idx'] + 50

# # combine df
# df_combined = pd.concat([df_block, df_rw], ignore_index=True)

# # save combined df
# df_combined.to_pickle('/root/capsule/code/TaskTrainedRNNData-combined_block_rw.pkl')

## test data loader for task_trained_rnn

In [ ]:
# # BlockBandit
# subject_id = 'BlockBandit2ArmCoupledEasy-v1-noisy_rl2_10__16:07_13:48:09'
# RandomWalkDF
subject_id = 'RandomWalkDF-v0-noisy_rl2_10__28:09_01:45:33'


loader = TaskTrainedRNNDatasetLoader(
    subject_ids=[subject_id],
    ignore_policy='exclude',
    features={'animal_response': 'prev choice', 'rewarded': 'prev reward'},
    eval_every_n=2,
    multisubject=False,
    seed=42,
)
dataset_bundle = loader.load()

## plot latents from trained dis-rnn

In [ ]:
from disentangled_rnns.library import disrnn, plotting, rnn_utils

In [ ]:
fig_dpi = 300

def plot_traj_in_PC_space(
    transformed_RNN_hidden_states_background,
    background_coloring_array,
    background_colorbar_label,
    background_coloring_minmax,
    num_pcs2plot,
    transformed_RNN_hidden_states_example_run=None,
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
):
    """
    Plot the transformed hidden states in the PCA space.
    Args:
        transformed_RNN_hidden_states_background: Transformed RNN hidden states for background
        transformed_RNN_hidden_states_example_run: Transformed RNN hidden states for an example run
        coloring_array: Array to color each state
        num_pcs2plot: Number of principal components to plot
        
    Returns:
        fig: matplotlib figure
    """
    n_axs = num_pcs2plot * (num_pcs2plot-1) / 2
    n_cols = num_pcs2plot
    n_rows = int(n_axs/ n_cols) + (n_axs% n_cols > 0)
    
    if n_rows == 1:
        n_rows = 2

    fig, axs = plt.subplots(
        nrows=n_rows, ncols=n_cols,
        figsize=(6*n_cols, 4*n_rows), dpi=fig_dpi
    )

    ax_id = 0
    for pc_x in range(num_pcs2plot):
        for pc_y in range(num_pcs2plot):
            if pc_x < pc_y:
                ax = axs[int(ax_id/n_cols), ax_id%n_cols]
                
                # set the title
                if transformed_fixed_points is not None:
                    ax.set_title(f'Fixed points under {fixed_points_condition}', fontsize=18)
                else:
                    ax.set_title(f'Neural space trajectoy', fontsize=18)
                
                # Plot the background
                scatter = ax.scatter(
                    transformed_RNN_hidden_states_background[:, :, pc_x],
                    transformed_RNN_hidden_states_background[:, :, pc_y],
                    s=2.5,
                    c=background_coloring_array, cmap=cm.coolwarm,
                    vmin=background_coloring_minmax[0], 
                    vmax=background_coloring_minmax[1],
                    alpha=0.9
                )
                ax.set_xlabel(f'PC {pc_x}', fontsize=18)
                ax.set_ylabel(f'PC {pc_y}', fontsize=18)
                ax.tick_params(axis='both', which='major', labelsize=12)
                fig.colorbar(
                    scatter, ax=ax,
                    label=background_colorbar_label
                )
                
                # Plot the trajectory of the example run
                if transformed_RNN_hidden_states_example_run is not None:
                    ax.plot(
                        transformed_RNN_hidden_states_example_run[:, pc_x],
                        transformed_RNN_hidden_states_example_run[:, pc_y],
                        color='k', lw=1, alpha=0.9
                    )
                    exameple_run_len = transformed_RNN_hidden_states_example_run.shape[0]
                    example_run = ax.scatter(
                        transformed_RNN_hidden_states_example_run[:, pc_x],
                        transformed_RNN_hidden_states_example_run[:, pc_y],
                        s=4,
                        c=np.arange(exameple_run_len), cmap=cm.copper,
                        vmin=0, vmax=exameple_run_len
                    )  # color coded by time step
                    fig.colorbar(
                        example_run, ax=ax,
                        label='Time step'
                    )

                # Plot the fixed points
                if transformed_fixed_points is not None:
                    scatter_fp = ax.scatter(
                        transformed_fixed_points[:, pc_x],
                        transformed_fixed_points[:, pc_y],
                        marker='o', s=5,
                        c=fixed_points_coloring_array, cmap=cm.gray,
                        vmin=1e-13, vmax=10, norm='log',
                        alpha=0.6
                    )  # color coded by q values
                    fig.colorbar(
                        scatter_fp, ax=ax,
                        label='Kinetic Energy (q value)'
                    )

                ax_id += 1

    fig.tight_layout()
    return fig

In [ ]:
import json

# BlockBandit
trained_model_fp = '/root/capsule/data/results-BlockBandit-251127-681834e2-9403-475d-8f2a-c4daa5c606d5'
# RandomWalkDF
# trained_model_fp = '/root/capsule/data/results-RandomWalkDF-251211-bb8c53d6-17d4-4160-badc-3bf1c27dab8e'
# combined
# trained_model_fp = '/root/capsule/data/results-combined-e73351b6-fba1-4b9f-bed9-8ddb35210a52'


model_id = '260116-2'


with open(
    f'{trained_model_fp}/{model_id}/inputs/jobs/.hydra/config.json', 
    'r'
) as fp:
    disrnn_config_json = json.load(fp)

with open(
    f'{trained_model_fp}/{model_id}/outputs/params.json', 
    'r'
) as fp:
    params = rnn_utils.to_np(json.load(fp))


# params_disrnn = params
params_disrnn = params['hk_disentangled_rnn']

In [ ]:
for key in params.keys():
    print(key)

In [ ]:
dataset_eval = dataset_bundle.eval_set

disrnn_config = disrnn.DisRnnConfig(
    # Dataset related
    obs_size=2,  # Choice, reward
    output_size=2,  # Choose left / choose right
    x_names=dataset_eval.x_names,
    y_names=dataset_eval.y_names,
    # Network architecture
    latent_size=disrnn_config_json['model']['architecture']['latent_size'],
    update_net_n_units_per_layer=disrnn_config_json['model']['architecture']['update_net_n_units_per_layer'],
    update_net_n_layers=disrnn_config_json['model']['architecture']['update_net_n_layers'],
    choice_net_n_units_per_layer=disrnn_config_json['model']['architecture']['choice_net_n_units_per_layer'],
    choice_net_n_layers=disrnn_config_json['model']['architecture']['choice_net_n_layers'],
    activation=disrnn_config_json['model']['architecture']['activation'],
    # Penalties
    # noiseless_mode=False,
    noiseless_mode=True,
    latent_penalty=disrnn_config_json['model']['penalties']['latent_penalty'],
    choice_net_latent_penalty=disrnn_config_json['model']['penalties']['choice_net_latent_penalty'],
    update_net_obs_penalty=disrnn_config_json['model']['penalties']['update_net_obs_penalty'],
    update_net_latent_penalty=disrnn_config_json['model']['penalties']['update_net_latent_penalty'],
    l2_scale=0,
)

In [ ]:
xs, ys = dataset_eval.get_all()
network_output, network_states = rnn_utils.eval_network(
    lambda: disrnn.HkDisentangledRNN(disrnn_config), params, xs)

# Compute normalized likelihood
logits = network_output[:,:,:2]  # First n_actions elements of network output are the logits (the final one is the penalty)
normalized_likelihood = rnn_utils.normalized_likelihood(labels = ys, output_logits=logits)

print(f'Normalized likelihood: {100*normalized_likelihood:.2f}%')

# Plot network activations on an example session
example_session = 0  # @param {type: "integer"}

choices = xs[:, example_session, 0]
rewards = xs[:, example_session, 1]
scalars = network_states[:, example_session, :]
# two_armed_bandits.plot_2ab_sessdata(choices,
#                                     rewards,
#                                     scalars=scalars,
#                                     scalar_types='agent_states',
#                                     show_legend=False)
# plt.plot(scalars)

In [ ]:
logit_sum = np.sum(np.exp(logits), axis=2)
action_prob = np.exp(logits)/ np.stack((logit_sum, logit_sum), axis=2)

In [ ]:
# plot open latents
open_latents = [0,2,4]
for latent_idx in open_latents:
    plt.plot(scalars[:, latent_idx], label=f'latent {latent_idx}')
plt.title('open latents')
plt.legend()


# # plot choice-relevant latents
# choice_latents = [0,4]
# for latent_idx in choice_latents:
#     plt.plot(scalars[:, latent_idx], label=f'latent {latent_idx}')
# plt.title('choice relevant latents')
# plt.legend()

# # plot choice-relevant latents
# non_choice_latents = [1,2,3]
# for latent_idx in non_choice_latents:
#     plt.plot(scalars[:, latent_idx], label=f'latent {latent_idx}')
# plt.title('Non choice relevant latents')
# plt.legend()

In [ ]:
n_sessions = network_states.shape[1]

fig = plot_traj_in_PC_space(
    network_states[:,:,open_latents],
    action_prob[:,:,1],
    'a1 prob',
    background_coloring_minmax=[0,1],
    num_pcs2plot=len(open_latents),
    transformed_RNN_hidden_states_example_run=network_states[:160,5,open_latents],
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
)

In [ ]:
# run pca on scalars and plot the first two pcs

# PCA using training set
pca_model = PCA(n_components=5)
pca_model.fit(network_states.reshape(-1, 5))
print(f'fitted disrnn, PCA explained variance (training): {pca_model.explained_variance_ratio_}')

# plot
fig, axs = plt.subplots(
    nrows=1, ncols=2,
    figsize=(6, 3), dpi=300
)

ax = axs[0]
ax.plot(pca_model.explained_variance_ratio_)
ax.set_title('fitted disrnn')
ax.set_xlabel('PC')
ax.set_ylabel('var. exp.')

ax = axs[1]
ax.plot(np.cumsum(pca_model.explained_variance_ratio_))
ax.set_title('cumulative var. explained')
ax.set_xlabel('PC')
ax.set_ylabel('cumulative var. exp.')
ax.set_ylim(0, 1)

fig.tight_layout()


In [ ]:
# PCA transformed training set
n_sessions = network_states.shape[1]
transformed_network_states = pca_model.transform(
    network_states.reshape(-1, 5)).\
    reshape(-1, n_sessions, 5)


# plt.scatter(
#     transformed_network_states[:,:,0],
#     transformed_network_states[:,:,1],
#     marker='o', s=5, alpha=0.85
# )
# plt.plot(
#     transformed_network_states[:160,0,0],
#     transformed_network_states[:160,0,1],
#     color='k', lw=1
# )


fig = plot_traj_in_PC_space(
    transformed_network_states,
    action_prob[:,:,1],
    'a1 prob',
    background_coloring_minmax=[0,1],
    num_pcs2plot=5,
    transformed_RNN_hidden_states_example_run=transformed_network_states[:160,5,:],
    transformed_fixed_points=None,
    fixed_points_coloring_array=None,
    fixed_points_condition=None
)